# Four-model GFL breaker load-step — Python API benchmark

Python/Jupyter equivalent of the C++ `EMT_Ph3_GFL_Four_Model_BreakerLoadStep` example.

The notebook runs the four converter implementations through `dpsimpy`:

1. `dpsimpy.emt.ph3.GFL`
2. `dpsimpy.emt.ph3.SSN_GFL`
3. `dpsimpy.emt.ph3.SSN_GFL_Split`
4. `dpsimpy.emt.ph3.AvVoltSourceInverterStateSpace`

All four cases use the same external EMT grid and the same SP power-flow initialization:

```text
GFL ---- R_grid ---- L_grid ---- NetworkInjection
  |
 PCC ---- breaker ---- nLoad ---- R_load ---- GND
```

The breaker is open initially and closes at **t = 5 s**.  
The added load is a balanced **5 kW resistive load**.

Matrix policy is kept identical to the C++ benchmark:

| Model | Matrix recomputation |
|---|---|
| GFL | OFF |
| SSN_GFL | ON |
| SSN_GFL_Split | OFF |
| AvVoltSourceInverterStateSpace | ON |

## 1. Imports and pybind sanity check

In [ ]:
from pathlib import Path
import time
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dpsimpy

print("Python API module:", dpsimpy.__file__)

required_classes = [
    "GFL",
    "SSN_GFL",
    "SSN_GFL_Split",
    "AvVoltSourceInverterStateSpace",
]

for cls_name in required_classes:
    cls = getattr(dpsimpy.emt.ph3, cls_name)
    print(f"{cls_name:34s}: {cls}")

## 2. Benchmark parameters

In [ ]:
# Simulation
TIME_STEP_EMT = 100e-6
FINAL_TIME_EMT = 10.0

# System
SYSTEM_FREQUENCY = 50.0
SYSTEM_OMEGA = 2.0 * np.pi * SYSTEM_FREQUENCY
V_GRID_LL_RMS = 400.0

# External grid
R_GRID = 0.3
L_GRID = 0.1e-3

# Converter filter
LF = 2e-3
CF = 10e-6
RF = 0.2
RC = 0.2  # only the three Rc-based state-space / SSN models

# Controller
KP_PLL = 0.25
KI_PLL = 0.2
OMEGA_CUTOFF = SYSTEM_OMEGA

P_REF_FILTER = 10_000.0
Q_REF_FILTER = 5_000.0

KP_POWER = 0.05
KI_POWER = 0.2
KP_CURRENT = 0.25
KI_CURRENT = 1.0

# Breaker load step
LOAD_STEP_TIME = 5.0
LOAD_STEP_P = 5_000.0
BREAKER_OPEN_R = 1e9
BREAKER_CLOSED_R = 1e-3

BASE_NAME = "PY_EMT_Ph3_GFL_Four_Model_BreakerLoadStep"
LOG_ROOT = Path("logs") / "python_api_gfl_comparison"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print(f"dt       = {TIME_STEP_EMT:g} s")
print(f"T_end    = {FINAL_TIME_EMT:g} s")
print(f"steps    = {round(FINAL_TIME_EMT / TIME_STEP_EMT):d}")
print(f"loadstep = {LOAD_STEP_P/1000:.1f} kW at t={LOAD_STEP_TIME:g} s")
print("log root =", LOG_ROOT)

## 3. Equivalent PCC reference for the new GFL

The three Rc-based models retain the capacitor-side reference
\(P_{vc}=10\,\mathrm{kW}\), \(Q_{vc}=5\,\mathrm{kvar}\).

The new composite `GFL` has no \(R_c\), so it uses the equivalent PCC operating point, exactly like the C++ example.

In [ ]:
def pcc_power_from_filter_reference(p_filter_ref, q_filter_ref):
    v_pcc_peak_phase = dpsimpy.RMS3PH_TO_PEAK1PH * V_GRID_LL_RMS

    if abs(RC) < 1e-12 or v_pcc_peak_phase < 1e-9:
        return p_filter_ref, q_filter_ref

    q_pcc_ref = q_filter_ref

    a = RC / (1.5 * v_pcc_peak_phase * v_pcc_peak_phase)

    discriminant = 1.0 + 4.0 * a * (p_filter_ref - a * q_pcc_ref * q_pcc_ref)

    if discriminant < 0.0:
        raise RuntimeError("No feasible PCC power for requested filter-side P/Q.")

    sqrt_disc = np.sqrt(discriminant)

    p1 = (-1.0 + sqrt_disc) / (2.0 * a)
    p2 = (-1.0 - sqrt_disc) / (2.0 * a)

    p_pcc_ref = p1 if abs(p1 - p_filter_ref) < abs(p2 - p_filter_ref) else p2

    return float(p_pcc_ref), float(q_pcc_ref)


P_PCC_REF, Q_PCC_REF = pcc_power_from_filter_reference(
    P_REF_FILTER,
    Q_REF_FILTER,
)

print(f"P_vc  = {P_REF_FILTER:.6f} W")
print(f"Q_vc  = {Q_REF_FILTER:.6f} var")
print(f"P_pcc = {P_PCC_REF:.6f} W")
print(f"Q_pcc = {Q_PCC_REF:.6f} var")

## 4. SP power-flow initialization

In [ ]:
def run_power_flow():
    sim_name = BASE_NAME + "_PF"
    log_dir = LOG_ROOT / sim_name
    log_dir.mkdir(parents=True, exist_ok=True)
    dpsimpy.Logger.set_log_dir(str(log_dir))

    # Nodes
    n_grid = dpsimpy.sp.SimNode(
        "nGrid",
        dpsimpy.PhaseType.Single,
    )
    n_series = dpsimpy.sp.SimNode(
        "nSeries",
        dpsimpy.PhaseType.Single,
    )
    n_pcc = dpsimpy.sp.SimNode(
        "nPcc",
        dpsimpy.PhaseType.Single,
    )

    # Slack
    slack = dpsimpy.sp.ph1.NetworkInjection(
        "Slack",
        dpsimpy.LogLevel.off,
    )
    slack.set_parameters(voltage_set_point=V_GRID_LL_RMS)
    slack.set_base_voltage(V_GRID_LL_RMS)
    slack.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.VD)

    # Separate PF branches are intentional so nSeries has a PF solution.
    resistor_pf = dpsimpy.sp.ph1.PiLine(
        "GridResistorPF",
        dpsimpy.LogLevel.off,
    )
    resistor_pf.set_parameters(
        R=R_GRID,
        L=0.0,
        C=0.0,
        G=0.0,
    )
    resistor_pf.set_base_voltage(V_GRID_LL_RMS)

    inductor_pf = dpsimpy.sp.ph1.PiLine(
        "GridInductorPF",
        dpsimpy.LogLevel.off,
    )
    inductor_pf.set_parameters(
        R=0.0,
        L=L_GRID,
        C=0.0,
        G=0.0,
    )
    inductor_pf.set_base_voltage(V_GRID_LL_RMS)

    # Converter represented as negative PQ load.
    inverter_pf = dpsimpy.sp.ph1.Load(
        "INV_GFL_PF",
        dpsimpy.LogLevel.off,
    )
    inverter_pf.set_parameters(
        active_power=-P_PCC_REF,
        reactive_power=-Q_PCC_REF,
        nominal_voltage=V_GRID_LL_RMS,
    )
    inverter_pf.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PQ)

    inverter_pf.connect([n_pcc])
    resistor_pf.connect([n_pcc, n_series])
    inductor_pf.connect([n_series, n_grid])
    slack.connect([n_grid])

    system_pf = dpsimpy.SystemTopology(
        SYSTEM_FREQUENCY,
        [n_grid, n_series, n_pcc],
        [slack, resistor_pf, inductor_pf, inverter_pf],
    )

    logger = dpsimpy.Logger(sim_name)
    logger.log_attribute("v_grid_pf", "v", n_grid)
    logger.log_attribute("v_series_pf", "v", n_series)
    logger.log_attribute("v_pcc_pf", "v", n_pcc)

    sim = dpsimpy.Simulation(
        sim_name,
        dpsimpy.LogLevel.off,
    )
    sim.set_system(system_pf)
    sim.set_time_step(1.0)
    sim.set_final_time(2.0)
    sim.set_domain(dpsimpy.Domain.SP)
    sim.set_solver(dpsimpy.Solver.NRP)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
    sim.do_init_from_nodes_and_terminals(False)
    sim.add_logger(logger)
    sim.run()

    print("PF initial voltages:")
    print("  nGrid  =", n_grid.initial_single_voltage())
    print("  nSeries=", n_series.initial_single_voltage())
    print("  nPcc   =", n_pcc.initial_single_voltage())

    return system_pf


system_pf = run_power_flow()

## 5. Common EMT topology helpers

The load step is a real breaker-connected resistor:

```text
PCC ---- SeriesSwitch ---- nLoad ---- SeriesResistor ---- GND
```

The resistor is sized from the PF-initialized PCC line-line RMS voltage.

In [ ]:
def create_breaker_load(n_pcc):
    n_load = dpsimpy.emt.SimNode(
        "nLoad",
        dpsimpy.PhaseType.ABC,
    )

    breaker = dpsimpy.emt.ph3.SeriesSwitch(
        "LoadBreaker",
        dpsimpy.LogLevel.off,
    )
    breaker.set_parameters(
        BREAKER_OPEN_R,
        BREAKER_CLOSED_R,
        False,
    )
    breaker.open()

    resistor = dpsimpy.emt.ph3.SeriesResistor(
        "LoadResistor",
        dpsimpy.LogLevel.off,
    )

    # Temporary value; replaced after PF transfer.
    resistor.set_parameters(V_GRID_LL_RMS**2 / LOAD_STEP_P)

    # Match the C++ orientation.
    breaker.connect([n_load, n_pcc])
    resistor.connect([dpsimpy.emt.SimNode.gnd, n_load])

    return n_load, breaker, resistor


def calculate_load_resistance(pcc_voltage_rms):
    v_pcc_ll_rms = abs(pcc_voltage_rms)

    if v_pcc_ll_rms < 1e-9:
        raise RuntimeError("Cannot calculate load resistance from zero PCC voltage.")

    # Balanced wye resistive load:
    # P_3ph = V_LL^2 / R_phase.
    return v_pcc_ll_rms**2 / LOAD_STEP_P


def create_common_grid():
    n_grid = dpsimpy.emt.SimNode(
        "nGrid",
        dpsimpy.PhaseType.ABC,
    )
    n_series = dpsimpy.emt.SimNode(
        "nSeries",
        dpsimpy.PhaseType.ABC,
    )
    n_pcc = dpsimpy.emt.SimNode(
        "nPcc",
        dpsimpy.PhaseType.ABC,
    )

    n_load, breaker, load_resistor = create_breaker_load(n_pcc)

    slack = dpsimpy.emt.ph3.NetworkInjection(
        "Slack",
        dpsimpy.LogLevel.off,
    )

    grid_resistor = dpsimpy.emt.ph3.Resistor(
        "GridResistor",
        dpsimpy.LogLevel.off,
    )
    grid_resistor.set_parameters(
        dpsimpy.Math.single_phase_parameter_to_three_phase(R_GRID)
    )

    grid_inductor = dpsimpy.emt.ph3.Inductor(
        "GridInductor",
        dpsimpy.LogLevel.off,
    )
    grid_inductor.set_parameters(
        dpsimpy.Math.single_phase_parameter_to_three_phase(L_GRID)
    )

    grid_resistor.connect([n_pcc, n_series])
    grid_inductor.connect([n_series, n_grid])
    slack.connect([n_grid])

    return {
        "n_grid": n_grid,
        "n_series": n_series,
        "n_pcc": n_pcc,
        "n_load": n_load,
        "breaker": breaker,
        "load_resistor": load_resistor,
        "slack": slack,
        "grid_resistor": grid_resistor,
        "grid_inductor": grid_inductor,
    }

## 6. Converter factory and common logging

In [ ]:
def create_inverter(model):
    if model == "GFL":
        inv = dpsimpy.emt.ph3.GFL(
            "INV_GFL",
            dpsimpy.LogLevel.off,
        )

        inv.set_parameters(
            SYSTEM_OMEGA,
            V_GRID_LL_RMS,
            P_PCC_REF,
            Q_PCC_REF,
        )

        inv.set_controller_parameters(
            KP_PLL,
            KI_PLL,
            KP_POWER,
            KI_POWER,
            KP_CURRENT,
            KI_CURRENT,
            OMEGA_CUTOFF,
        )

        inv.set_filter_parameters(
            LF,
            CF,
            RF,
        )

        inv.with_control(True)
        return inv

    cls = {
        "SSN_GFL": dpsimpy.emt.ph3.SSN_GFL,
        "SSN_GFL_Split": dpsimpy.emt.ph3.SSN_GFL_Split,
        "AvVoltSourceInverterStateSpace": dpsimpy.emt.ph3.AvVoltSourceInverterStateSpace,
    }[model]

    # SSN_GFL and SSN_GFL_Split expose the uid/name constructor.
    if model in {"SSN_GFL", "SSN_GFL_Split"}:
        inv = cls(
            "INV_GFL",
            "INV_GFL",
            dpsimpy.LogLevel.off,
        )
    else:
        inv = cls(
            "INV_GFL",
            dpsimpy.LogLevel.off,
        )

    inv.set_parameters(
        LF,
        CF,
        RF,
        RC,
        SYSTEM_OMEGA,
        KP_PLL,
        KI_PLL,
        OMEGA_CUTOFF,
        P_REF_FILTER,
        Q_REF_FILTER,
        KP_POWER,
        KI_POWER,
        KP_CURRENT,
        KI_CURRENT,
    )

    return inv


def add_common_logging(logger, grid, inverter, model):
    logger.log_attribute(
        "v_grid",
        "v",
        grid["n_grid"],
    )
    logger.log_attribute(
        "v_series",
        "v",
        grid["n_series"],
    )
    logger.log_attribute(
        "v_pcc",
        "v",
        grid["n_pcc"],
    )

    logger.log_attribute(
        "v_load",
        "v",
        grid["n_load"],
    )
    logger.log_attribute(
        "i_breaker",
        "i_intf",
        grid["breaker"],
    )
    logger.log_attribute(
        "i_load",
        "i_intf",
        grid["load_resistor"],
    )

    logger.log_attribute(
        "i_inv",
        "i_intf",
        inverter,
    )

    logger.log_attribute(
        "vc_d",
        "vc_d",
        inverter,
    )
    logger.log_attribute(
        "vc_q",
        "vc_q",
        inverter,
    )

    if model == "GFL":
        logger.log_attribute(
            "igrid_d",
            "igrid_d",
            inverter,
        )
        logger.log_attribute(
            "igrid_q",
            "igrid_q",
            inverter,
        )
    else:
        logger.log_attribute(
            "igrid_d",
            "irc_d",
            inverter,
        )
        logger.log_attribute(
            "igrid_q",
            "irc_q",
            inverter,
        )

    logger.log_attribute(
        "omega_pll",
        "omega_pll",
        inverter,
    )

## 7. Run one EMT model

In [ ]:
MATRIX_RECOMPUTATION = {
    "GFL": False,
    "SSN_GFL": True,
    "SSN_GFL_Split": False,
    "AvVoltSourceInverterStateSpace": True,
}


def run_emt_case(model):
    sim_name = f"{BASE_NAME}_{model}"

    log_dir = LOG_ROOT / sim_name
    log_dir.mkdir(parents=True, exist_ok=True)
    dpsimpy.Logger.set_log_dir(str(log_dir))

    grid = create_common_grid()
    inverter = create_inverter(model)

    if model == "GFL":
        inverter.connect(
            [
                grid["n_pcc"],
            ]
        )
    else:
        inverter.connect(
            [
                dpsimpy.emt.SimNode.gnd,
                grid["n_pcc"],
            ]
        )

    system = dpsimpy.SystemTopology(
        SYSTEM_FREQUENCY,
        [
            grid["n_grid"],
            grid["n_series"],
            grid["n_pcc"],
            grid["n_load"],
        ],
        [
            grid["slack"],
            grid["grid_resistor"],
            grid["grid_inductor"],
            inverter,
            grid["breaker"],
            grid["load_resistor"],
        ],
    )

    # Transfer nGrid / nSeries / nPcc PF operating point.
    system.init_with_powerflow(
        system_pf,
        dpsimpy.Domain.EMT,
    )

    load_resistance = calculate_load_resistance(grid["n_pcc"].initial_single_voltage())
    grid["load_resistor"].set_parameters(load_resistance)

    logger = dpsimpy.Logger(sim_name)
    add_common_logging(
        logger,
        grid,
        inverter,
        model,
    )

    sim = dpsimpy.Simulation(
        sim_name,
        dpsimpy.LogLevel.off,
    )
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(dpsimpy.Domain.EMT)
    sim.set_solver(dpsimpy.Solver.MNA)

    sim.do_system_matrix_recomputation(MATRIX_RECOMPUTATION[model])
    sim.do_init_from_nodes_and_terminals(True)

    sim.set_time_step(TIME_STEP_EMT)
    sim.set_final_time(FINAL_TIME_EMT)

    breaker_event = dpsimpy.event.SwitchEvent(
        LOAD_STEP_TIME,
        grid["breaker"],
        True,
    )
    sim.add_event(breaker_event)

    print(
        f"Running {model:34s} "
        f"[matrix recomputation="
        f"{'ON' if MATRIX_RECOMPUTATION[model] else 'OFF'}, "
        f"breaker closes at {LOAD_STEP_TIME:g} s]"
    )

    t0 = time.perf_counter()
    sim.run()
    wall_seconds = time.perf_counter() - t0

    csv_path = log_dir / f"{sim_name}.csv"

    print(f"  wall time : {wall_seconds:.6f} s")
    print(f"  R_load    : {load_resistance:.9f} Ohm")
    print(f"  CSV       : {csv_path}")

    return {
        "model": model,
        "simulation_name": sim_name,
        "wall_seconds": wall_seconds,
        "matrix_recomputation": MATRIX_RECOMPUTATION[model],
        "load_resistance": load_resistance,
        "csv": csv_path,
    }

## 8. Run all four Python API cases

This cell is the actual Python-side equivalent of the four sequential C++ runs.

At `dt = 100 µs` and `T_end = 10 s`, each case performs 100,000 EMT steps, so this cell can take a while.

In [ ]:
MODELS = [
    "GFL",
    "SSN_GFL",
    "SSN_GFL_Split",
    "AvVoltSourceInverterStateSpace",
]

results = {}

for model in MODELS:
    results[model] = run_emt_case(model)

print("\nFinished all four Python API simulations.")

## 9. Timing summary
Reported wall times include logging.

In [ ]:
steps = round(FINAL_TIME_EMT / TIME_STEP_EMT)
gfl_time = results["GFL"]["wall_seconds"]

timing_rows = []

for model in MODELS:
    r = results[model]
    wall = r["wall_seconds"]

    timing_rows.append(
        {
            "Model": model,
            "time [s]": wall,
            "us/step": 1e6 * wall / steps,
            "vs GFL": gfl_time / wall,
            "matrix": ("recompute" if r["matrix_recomputation"] else "fixed"),
        }
    )

timing_df = pd.DataFrame(timing_rows)
display(timing_df)

## 10. Load Python-generated CSVs

In [ ]:
def load_dpsim_csv(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(
        path,
        skipinitialspace=True,
    )
    df.columns = [str(c).strip() for c in df.columns]

    if "time" not in df.columns:
        raise ValueError(f"{path.name}: no 'time' column found")

    return df


runs = {model: load_dpsim_csv(results[model]["csv"]) for model in MODELS}

for model, df in runs.items():
    print(f"{model:34s}: " f"{df.shape[0]:8d} rows, " f"{df.shape[1]:3d} columns")

## 11. Signal/magnitude helpers

The plotting logic is the same as in the earlier four-model magnitude-comparison notebook:

\[
|x_{abc}| = \sqrt{x_a^2+x_b^2+x_c^2}
\]

\[
|x_{dq}| = \sqrt{x_d^2+x_q^2}
\]

In [ ]:
def _vector_columns(df, base):
    cols = [f"{base}_{i}" for i in range(3)]
    if all(c in df.columns for c in cols):
        return cols

    for suffixes in [
        ("_a", "_b", "_c"),
        (".a", ".b", ".c"),
    ]:
        cols = [base + suffix for suffix in suffixes]
        if all(c in df.columns for c in cols):
            return cols

    return None


def abc_magnitude(df, base):
    cols = _vector_columns(
        df,
        base,
    )

    if cols is None:
        raise KeyError(
            f"Could not find 3-phase columns for '{base}'. "
            f"First columns: {list(df.columns)[:25]}"
        )

    x = df[cols].to_numpy(dtype=float)

    return np.sqrt(np.sum(x * x, axis=1))


def dq_magnitude(df, d_col, q_col):
    if d_col not in df.columns or q_col not in df.columns:
        raise KeyError(f"Missing {d_col}/{q_col}")

    d = df[d_col].to_numpy(dtype=float)
    q = df[q_col].to_numpy(dtype=float)

    return np.sqrt(d * d + q * q)


def time_mask(time_values, start=None, end=None):
    mask = np.ones(
        len(time_values),
        dtype=bool,
    )

    if start is not None:
        mask &= time_values >= start

    if end is not None:
        mask &= time_values <= end

    return mask


def plot_comparison(
    quantity_fn,
    title,
    ylabel,
    start=None,
    end=None,
    downsample=1,
    event_time=LOAD_STEP_TIME,
):
    plt.figure(figsize=(14, 5))

    for model, df in runs.items():
        t = df["time"].to_numpy(dtype=float)
        y = np.asarray(
            quantity_fn(df),
            dtype=float,
        )

        valid = np.isfinite(t) & np.isfinite(y)
        valid &= time_mask(
            t,
            start,
            end,
        )

        idx = np.flatnonzero(valid)[::downsample]

        plt.plot(
            t[idx],
            y[idx],
            linewidth=1.1,
            label=model,
        )

    if (
        event_time is not None
        and (start is None or event_time >= start)
        and (end is None or event_time <= end)
    ):
        plt.axvline(
            event_time,
            linestyle="--",
            linewidth=0.9,
            label="breaker close",
        )

    plt.title(title)
    plt.xlabel("Time [s]")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 12. Breaker/load sanity check

In [ ]:
plot_comparison(
    lambda df: abc_magnitude(df, "i_breaker"),
    "|i_breaker,abc| comparison",
    "Breaker current vector magnitude [A]",
    start=4.9,
    end=5.1,
)

plot_comparison(
    lambda df: abc_magnitude(df, "i_load"),
    "|i_load,abc| comparison",
    "Load current vector magnitude [A]",
    start=4.9,
    end=5.1,
)

## 13. All comparison quantities

In [ ]:
def plot_all_magnitudes(
    start=None,
    end=None,
    downsample=1,
):
    comparisons = [
        (
            lambda df: abc_magnitude(
                df,
                "v_pcc",
            ),
            "|v_pcc,abc| comparison",
            "Voltage vector magnitude [V]",
        ),
        (
            lambda df: abc_magnitude(
                df,
                "v_grid",
            ),
            "|v_grid,abc| comparison",
            "Voltage vector magnitude [V]",
        ),
        (
            lambda df: abc_magnitude(
                df,
                "v_series",
            ),
            "|v_series,abc| comparison",
            "Voltage vector magnitude [V]",
        ),
        (
            lambda df: abc_magnitude(
                df,
                "i_inv",
            ),
            "|i_inv,abc| comparison",
            "Current vector magnitude [A]",
        ),
        (
            lambda df: abc_magnitude(
                df,
                "i_load",
            ),
            "|i_load,abc| comparison",
            "Load current vector magnitude [A]",
        ),
        (
            lambda df: dq_magnitude(
                df,
                "vc_d",
                "vc_q",
            ),
            "|v_c,dq| comparison",
            "Voltage dq magnitude [V]",
        ),
        (
            lambda df: dq_magnitude(
                df,
                "igrid_d",
                "igrid_q",
            ),
            "|i_grid,dq| comparison",
            "Grid-current dq magnitude [A]",
        ),
        (
            lambda df: (df["omega_pll"].to_numpy(dtype=float) / (2.0 * np.pi)),
            "PLL frequency comparison",
            "Frequency [Hz]",
        ),
    ]

    for fn, title, ylabel in comparisons:
        plot_comparison(
            fn,
            title,
            ylabel,
            start=start,
            end=end,
            downsample=downsample,
        )

## 14. Transient comparison around the 5 s load step

This is the most useful view for checking dynamic equivalence of the four implementations.

In [ ]:
plot_all_magnitudes(
    start=4.9,
    end=5.5,
    downsample=1,
)

## 15. Optional full 10 s comparison

Uncomment if you also want the complete simulation window.

In [ ]:
# plot_all_magnitudes(
#     start=0.0,
#     end=10.0,
#     downsample=10,
# )

## 16. Optional numerical spread between models

Useful when the curves visually lie almost exactly on top of one another.

In [ ]:
def aligned_values(quantity_fn):
    reference_model = MODELS[0]
    reference_time = runs[reference_model]["time"].to_numpy(dtype=float)

    values = {}

    for model, df in runs.items():
        t = df["time"].to_numpy(dtype=float)
        y = np.asarray(
            quantity_fn(df),
            dtype=float,
        )

        if len(t) == len(reference_time) and np.allclose(
            t,
            reference_time,
            rtol=0.0,
            atol=1e-12,
            equal_nan=True,
        ):
            values[model] = y
        else:
            good = np.isfinite(t) & np.isfinite(y)

            values[model] = np.interp(
                reference_time,
                t[good],
                y[good],
            )

    return reference_time, values


def print_spread(name, quantity_fn):
    t, values = aligned_values(quantity_fn)

    stack = np.vstack(list(values.values()))

    finite = np.all(
        np.isfinite(stack),
        axis=0,
    )

    spread = np.max(stack[:, finite], axis=0) - np.min(stack[:, finite], axis=0)

    if spread.size == 0:
        print(f"{name:24s}: " "no common finite samples")
        return

    print(
        f"{name:24s}: "
        f"max={np.max(spread):.6e}, "
        f"RMS={np.sqrt(np.mean(spread**2)):.6e}"
    )


print_spread(
    "v_pcc magnitude",
    lambda df: abc_magnitude(
        df,
        "v_pcc",
    ),
)

print_spread(
    "i_inv magnitude",
    lambda df: abc_magnitude(
        df,
        "i_inv",
    ),
)

print_spread(
    "vc dq magnitude",
    lambda df: dq_magnitude(
        df,
        "vc_d",
        "vc_q",
    ),
)

print_spread(
    "igrid dq magnitude",
    lambda df: dq_magnitude(
        df,
        "igrid_d",
        "igrid_q",
    ),
)

print_spread(
    "PLL frequency",
    lambda df: (df["omega_pll"].to_numpy(dtype=float) / (2.0 * np.pi)),
)